# 11. Graph Databases

Store a tiny TGraph as RDF triples, ask a SPARQL relationship query, and save the dataset locally. This is a gateway to RDF graph databases without a server or account.

**Goal:** Complete one small workflow. Run the cells from top to bottom in a fresh kernel.

**Extra dependencies:** `rdflib==7.2.1`. See the setup instructions before running.

All examples use metres unless stated otherwise.


In [ ]:
import os
os.environ["TOPOLOGICPY_CORE_BACKEND"] = "pythonocc"

import topologicpy
from topologicpy.Core import Core

assert topologicpy.__version__ == "0.9.86", "Use TopologicPy 0.9.86 for these tutorials."
assert type(Core.Backend()).__name__ == "PythonOCCBackend", "Restart with the pythonocc backend."
print("TopologicPy", topologicpy.__version__, "—", type(Core.Backend()).__name__)


## Build the graph


In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.TGraph import TGraph

entrance = Vertex.ByCoordinates(0, 0, 0)
hall = Vertex.ByCoordinates(2, 0, 0)
studio = Vertex.ByCoordinates(4, 0, 0)
office = Vertex.ByCoordinates(2, 2, 0)
vertices = [entrance, hall, studio, office]
edges = [{"src": 0, "dst": 1},
         {"src": 1, "dst": 2},
         {"src": 1, "dst": 3}]
graph = TGraph.ByVerticesEdges(vertices, edges, directed=False)
assert isinstance(graph, TGraph)
assert len(TGraph.Vertices(graph)) == 4
assert len(TGraph.Edges(graph)) == 3


## Map TGraph data into an RDF dataset

An RDF triple has a subject, predicate and object. A vertex becomes an identified resource; an edge becomes a `connects` relationship. `Dataset` is a storage/query container here: the spatial graph itself is created only with `TGraph`. This example uses a local in-memory store, rather than a production database service.


In [ ]:
from rdflib import Dataset, Namespace, Literal, RDF

store = Dataset(default_union=True)
space = Namespace("https://example.org/tutorial/")
names = ["Entrance", "Hall", "Studio", "Office"]
mesh = TGraph.MeshData(graph)
for index in range(len(mesh["vertices"])):
    node = space[str(index)]
    store.add((node, RDF.type, space.Space))
    store.add((node, space.name, Literal(names[index])))
for start, end in mesh["edges"]:
    store.add((space[str(start)], space.connects, space[str(end)]))
    store.add((space[str(end)], space.connects, space[str(start)]))


## Query and save

SPARQL matches triple patterns. The hall is vertex 1; its neighbours should be the entrance, office and studio. TriG preserves the RDF dataset on disk and can be imported into a larger RDF database.


In [ ]:
from pathlib import Path

query = """PREFIX s: <https://example.org/tutorial/>
SELECT ?name WHERE { <https://example.org/tutorial/1> s:connects ?other .
                     ?other s:name ?name . } ORDER BY ?name"""
neighbours = []
for row in store.query(query):
    neighbours.append(str(row[0]))
print("Connected to Hall:", neighbours)
assert neighbours == ["Entrance", "Office", "Studio"]
output = Path("results")
output.mkdir(exist_ok=True)
store.serialize(destination=str(output / "11_spaces.trig"), format="trig")
restored = Dataset(default_union=True)
restored.parse(str(output / "11_spaces.trig"), format="trig")
assert len(list(restored.query(query))) == 3


## Try it yourself

Add coordinates as RDF properties, or create a second TGraph with one more connection and compare query results.
